In [ ]:
import csv
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn, optim
from torchvision import datasets, transforms

sys.path.append(os.path.abspath(".."))
from src.architectures import GeneralMLP
from src.continual_learning import GPM
from src.utils import apply_heavy_tailed_init, set_seed

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- 1. GPU-Accelerated Data Loading ---
def generate_permutations(num_tasks, num_pixels=784, seed=42):
    """Generates clean pseudorandom domain permutations mapping each task stream."""
    rng = np.random.RandomState(seed)
    perms = [
        torch.arange(num_pixels).to(DEVICE)
    ]  # Task 0 is standard un-permuted MNIST
    for _ in range(num_tasks - 1):
        perms.append(torch.from_numpy(rng.permutation(num_pixels)).to(DEVICE))
    return perms


def get_gpu_data(dataset):
    """Loads raw tensors to GPU once to avoid repetitive overhead."""
    imgs = torch.stack([img for img, _ in dataset]).to(DEVICE).view(-1, 784)
    lbls = torch.tensor([lbl for _, lbl in dataset]).to(DEVICE)
    return imgs, lbls


def save_physics_snapshot(model, input_batch, output_dir, t_idx, epoch, alpha, g):
    model.eval()

    # Capture pre-activations
    pre_acts = model.get_pre_activations(input_batch)
    linear_layers = [m for m in model.modules() if isinstance(m, nn.Linear)]

    layer_physics = {}
    for idx, layer in enumerate(linear_layers):
        layer_key = f"layer_{idx}" if idx < len(linear_layers) - 1 else "classifier"
        # Extract pre-activation for this specific layer
        h = pre_acts[idx] if idx < len(linear_layers) - 1 else pre_acts["classifier"]

        layer_physics[layer_key] = {"pre_activations": h.float().cpu()}

    snapshot = {
        "metadata": {"task": t_idx + 1, "epoch": epoch, "alpha": alpha, "g": g},
        "state_dict": model.state_dict(),  # Weights W are stored here
        "physics_data": layer_physics,  # Only storing h to save space
    }

    output_dir.mkdir(parents=True, exist_ok=True)
    file_path = output_dir / f"snapshot_T{t_idx + 1}_E{epoch}.pt"

    # Using weights + pre_acts allows reconstruction of J, Rank, and CKA later
    torch.save(snapshot, file_path)
    return file_path


class CLMetricsTracker:
    def __init__(self, max_tasks=20):
        self.max_tasks = max_tasks
        # defaultdict-style dynamic storage to avoid rigid pre-allocation
        self.history = {}
        self._total_steps_logged = 0

    def _ensure_key_exists(self, key):
        """Lazy initialization for new dynamic metric keys."""
        if key not in self.history:
            # Pad retroactively with None for previous steps if a metric is added late
            self.history[key] = [None] * self._total_steps_logged

    def log(self, step, acc_list, **extra_metrics):
        """
        Logs a single evaluation step.

        Args:
            step: Global step index.
            acc_list: List of task accuracies [acc_t0, acc_t1, ...]
            **extra_metrics: Arbitrary method-specific key-value pairs.
                             Examples:
                               basis_rank=[12, 18, 5]
                               cum_rank=35
                               effective_rank={"layer1": 4.2, "layer2": 8.1}
        """
        self._ensure_key_exists("step")
        self.history["step"].append(step)

        # 1. Log Task Accuracies
        for t_idx in range(self.max_tasks):
            col_key = f"task_{t_idx}_acc"
            self._ensure_key_exists(col_key)
            if t_idx < len(acc_list):
                self.history[col_key].append(float(acc_list[t_idx]))
            else:
                self.history[col_key].append(None)

        # 2. Dynamically Log Extra Method-Specific Metrics
        for metric_name, val in extra_metrics.items():
            if isinstance(val, (list, tuple)):
                # Handle sequence inputs (e.g., basis_rank per task or per layer)
                for idx, sub_val in enumerate(val):
                    sub_key = f"{metric_name}_{idx}"
                    self._ensure_key_exists(sub_key)
                    self.history[sub_key].append(
                        None if sub_val is None else float(sub_val)
                    )
            elif isinstance(val, dict):
                # Handle dictionary inputs (e.g., {"layer1": 4.5, "layer2": 8.2})
                for dict_key, sub_val in val.items():
                    sub_key = f"{metric_name}_{dict_key}"
                    self._ensure_key_exists(sub_key)
                    self.history[sub_key].append(
                        None if sub_val is None else float(sub_val)
                    )
            else:
                # Handle single scalar values
                self._ensure_key_exists(metric_name)
                self.history[metric_name].append(None if val is None else float(val))

        # 3. Pad any keys that were created previously but NOT passed in this log call
        self._total_steps_logged += 1
        for key in self.history:
            if len(self.history[key]) < self._total_steps_logged:
                self.history[key].append(None)

    def save_to_csv(self, filepath="cl_experiment_metrics.csv"):
        """Exports all recorded columns into a clean, flat CSV file."""
        directory = os.path.dirname(filepath)
        if directory and not os.path.exists(directory):
            os.makedirs(directory)

        # Gather headers dynamically, prioritizing 'step' first
        headers = ["step"] + [k for k in self.history if k != "step"]

        # Drop columns that are completely empty (all None)
        active_headers = [
            h for h in headers if any(val is not None for val in self.history[h])
        ]

        num_rows = self._total_steps_logged

        with open(filepath, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(active_headers)

            for r_idx in range(num_rows):
                row_data = [
                    "" if self.history[h][r_idx] is None else self.history[h][r_idx]
                    for h in active_headers
                ]
                writer.writerow(row_data)

        print(f"Metrics successfully exported to: '{filepath}'")

In [ ]:
import os
from pathlib import Path

import torch

# --- 1. GLOBAL EXPERIMENT HYPERPARAMETERS ---
RUN_SEEDS = [2, 3, 4]
NUM_TASKS = 20
EPOCHS_PER_TASK = 5
LR_TASK_0 = 1e-2  # Higher LR to rapidly align the shared backbone
LR_SUBSEQUENT = 1e-3  # Lower LR to prevent drift and preserve historical memory
BATCH_SIZE = 256
CALIB_SAMPLE_SIZE = 1024

ALPHA_INIT = 1.2
G_INIT = 1.0

# Structural variables matching target deep architecture
hidden_size = 784
depth = 9
activation_name = "tanh"
bias = False

# Algorithm Configuration
GLOBAL_THRESHOLD = 0.97
ORTHOG_METHOD = "qr"

CUSTOM_MASKING_FN = None
EVAL_EVERY_N_BATCHES = 100

# --- 2. OPTIMIZED DATA LOADING ---
print(f"Initializing VRAM memory pinning pipeline on device: {DEVICE}")
mnist_train = datasets.MNIST(
    "../data", train=True, download=True, transform=transforms.ToTensor()
)
mnist_test = datasets.MNIST(
    "../data", train=False, download=True, transform=transforms.ToTensor()
)

train_imgs, train_lbls = get_gpu_data(mnist_train)
test_imgs_raw, test_lbls = get_gpu_data(mnist_test)


# --- 3. CORE BACKGROUND SWEEP PIPELINE ---
for current_seed in RUN_SEEDS:
    print("\n" + "=" * 80)
    print(f"LAUNCHING PARAMETER SWEEP FOR RANDOM EXPERIMENTAL SEED [s={current_seed}]")
    print("=" * 80)

    set_seed(current_seed)
    task_permutations = generate_permutations(num_tasks=NUM_TASKS, seed=current_seed)

    model = GeneralMLP(784, hidden_size, 10, depth, activation_name, bias=bias).to(
        DEVICE
    )
    model = apply_heavy_tailed_init(
        model=model, alpha=ALPHA_INIT, g=G_INIT, seed=current_seed
    )

    optimizer = optim.SGD(model.parameters(), lr=LR_TASK_0)
    criterion = nn.CrossEntropyLoss()
    tracker = CLMetricsTracker(max_tasks=NUM_TASKS)

    gpm = GPM(variance_threshold=GLOBAL_THRESHOLD, orthog_method=ORTHOG_METHOD)

    linear_layer_info = [
        (name + ".weight", module)
        for name, module in model.named_modules()
        if isinstance(module, nn.Linear)
    ]

    total_steps = 0

    print(f"Architecture Depth: {depth} layers | Hidden Width: {hidden_size} neurons")
    print(
        f"GPM Mode: {'Standard GPM' if CUSTOM_MASKING_FN is None else 'Custom Masked GPM'}"
    )
    print(
        f"Orthogonalization Method: {ORTHOG_METHOD.upper()} | Target Energy: {GLOBAL_THRESHOLD * 100:.1f}%"
    )
    print("-" * 80)

    # --- 4. MAIN CONTINUAL LEARNING STREAM ---
    for t_idx in range(NUM_TASKS):
        # Dynamically set Learning Rate: Task 0 vs Subsequent Tasks
        current_lr = LR_TASK_0 if t_idx == 0 else LR_SUBSEQUENT
        for param_group in optimizer.param_groups:
            param_group["lr"] = current_lr

        current_perm = task_permutations[t_idx]

        tx = train_imgs[:, current_perm]
        ty = train_lbls

        print(
            f"\n--- Task {t_idx:02d}/{NUM_TASKS:02d} | Seed: {current_seed} | Active LR: {current_lr:.1e} ---"
        )

        for epoch in range(EPOCHS_PER_TASK):
            # ... rest of your epoch loop stays identical ...
            model.train()
            indices = torch.randperm(len(tx))

            for i in range(0, len(tx), BATCH_SIZE):
                batch_idx = indices[i : i + BATCH_SIZE]
                bx, by = tx[batch_idx], ty[batch_idx]

                optimizer.zero_grad()

                output = model(bx)
                loss_current = criterion(output, by)
                loss_current.backward()

                gpm.project_model_gradients(model)
                optimizer.step()

                # Snapshot logging evaluation cycle
                if total_steps % EVAL_EVERY_N_BATCHES == 0:
                    current_accs = []
                    model.eval()
                    with torch.no_grad():
                        for eval_t_idx in range(t_idx + 1):
                            eval_perm = task_permutations[eval_t_idx]
                            test_x = test_imgs_raw[:, eval_perm]

                            outputs = model(test_x[:1000])
                            preds = outputs.argmax(dim=1)
                            acc = (preds == test_lbls[:1000]).float().mean().item()
                            current_accs.append(acc)

                    # Log task accuracy along with per-layer basis ranks and total rank
                    tracker.log(
                        step=total_steps,
                        acc_list=current_accs,
                        basis_rank=gpm.get_basis_ranks(),
                        total_basis_rank=gpm.get_total_basis_rank(),
                    )

                    acc_report = " | ".join(
                        [
                            f"T{j}: {current_accs[j] * 100:.1f}%"
                            for j in range(t_idx + 1)
                        ]
                    )
                    print(
                        f"Seed {current_seed} | Step {total_steps:04d} (Epoch {epoch}) -> {acc_report} | Total Basis Rank: {gpm.get_total_basis_rank()}"
                    )

                total_steps += 1

        # --- 5. POST-TASK CALIBRATION STEP ---
        model.eval()
        calib_indices = torch.randperm(len(tx))[:CALIB_SAMPLE_SIZE]
        calib_images = tx[calib_indices]

        with torch.no_grad():
            layer_inputs_dict = model.get_layer_inputs(calib_images)

        for (weight_param_name, _), (layer_key, layer_input) in zip(
            linear_layer_info, layer_inputs_dict.items()
        ):
            added_rank = gpm.update_basis(
                layer_id=weight_param_name,
                live_activations=layer_input,
                masking_fn=CUSTOM_MASKING_FN,
            )

        print(
            f"Task {t_idx} complete. Subspace memory updated. Current total basis rank: {gpm.get_total_basis_rank()}"
        )

    # --- 6. END-OF-TRAINING SNAPSHOT & SERIALIZATION ---
    output_dir = Path("./checkpoints")
    final_calib_indices = torch.randperm(len(tx))[:CALIB_SAMPLE_SIZE]
    final_batch = tx[final_calib_indices]

    # snapshot_file = save_physics_snapshot(
    #     model=model,
    #     input_batch=final_batch,
    #     output_dir=output_dir,
    #     t_idx=NUM_TASKS - 1,
    #     epoch=EPOCHS_PER_TASK - 1,
    #     alpha=ALPHA_INIT,
    #     g=G_INIT,
    # )
    # print(f"End-of-training physics snapshot successfully saved to: {snapshot_file}")

    output_filename = f"schedule_gpm_a{ALPHA_INIT}_run_s{current_seed}.csv"
    tracker.save_to_csv(output_filename)
    print(
        f"Sweep for seed {current_seed} serialized successfully to {output_filename}.\n"
    )

print("\n" + "=" * 80)
print("ALL MULTI-SEED PARAMETER CONVERSIONS COMPLETED IN BACKGROUND POOL.")
print("=" * 80)

In [ ]:
import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

# --- 1. GLOBAL EXPERIMENT HYPERPARAMETERS ---
FIXED_SEED = 0
NUM_TASKS = 20
EPOCHS_PER_TASK = 5

# Task 0 LR Sweep Configuration (18 log-spaced values from 1e-4 to 0.2)
TASK1_LRS = [
    0.0090,
    0.0095,
    0.0098,
    0.0100,
    0.0102,
    0.0105,
    0.0108,
    0.0110,
]
LR_SUBSEQUENT = 1e-3  # Fixed LR for Tasks 1 through 19

BATCH_SIZE = 256
CALIB_SAMPLE_SIZE = 1024

ALPHA_INIT = 1.2
G_INIT = 1.0

# Structural variables matching target deep architecture
hidden_size = 784
depth = 9
activation_name = "tanh"
bias = False

# Algorithm Configuration
GLOBAL_THRESHOLD = 0.97
ORTHOG_METHOD = "qr"

CUSTOM_MASKING_FN = None
EVAL_EVERY_N_BATCHES = 100

# Output directory setup
OUTPUT_DIR = Path("./task1_lr_micro_sweep")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- 2. OPTIMIZED DATA LOADING ---
print(f"Initializing VRAM memory pinning pipeline on device: {DEVICE}")
mnist_train = datasets.MNIST(
    "../data", train=True, download=True, transform=transforms.ToTensor()
)
mnist_test = datasets.MNIST(
    "../data", train=False, download=True, transform=transforms.ToTensor()
)

train_imgs, train_lbls = get_gpu_data(mnist_train)
test_imgs_raw, test_lbls = get_gpu_data(mnist_test)


# --- 3. CORE BACKGROUND SWEEP PIPELINE ---
for lr_task0 in TASK1_LRS:
    lr_task0 = float(lr_task0)
    print("\n" + "=" * 80)
    print(
        f"LAUNCHING PARAMETER SWEEP FOR TASK 0 LEARNING RATE [LR_Task0={lr_task0:.5g}]"
    )
    print("=" * 80)

    set_seed(FIXED_SEED)
    task_permutations = generate_permutations(
        num_tasks=NUM_TASKS, seed=FIXED_SEED
    )

    model = GeneralMLP(
        784, hidden_size, 10, depth, activation_name, bias=bias
    ).to(DEVICE)
    model = apply_heavy_tailed_init(
        model=model, alpha=ALPHA_INIT, g=G_INIT, seed=FIXED_SEED
    )

    optimizer = optim.SGD(model.parameters(), lr=lr_task0)
    criterion = nn.CrossEntropyLoss()
    tracker = CLMetricsTracker(max_tasks=NUM_TASKS)

    gpm = GPM(variance_threshold=GLOBAL_THRESHOLD, orthog_method=ORTHOG_METHOD)

    linear_layer_info = [
        (name + ".weight", module)
        for name, module in model.named_modules()
        if isinstance(module, nn.Linear)
    ]

    total_steps = 0

    print(
        f"Architecture Depth: {depth} layers | Hidden Width: {hidden_size} neurons"
    )
    print(
        f"GPM Mode: {'Standard GPM' if CUSTOM_MASKING_FN is None else 'Custom Masked GPM'}"
    )
    print(
        f"Orthogonalization Method: {ORTHOG_METHOD.upper()} | Target Energy: {GLOBAL_THRESHOLD * 100:.1f}%"
    )
    print("-" * 80)

    # --- 4. MAIN CONTINUAL LEARNING STREAM ---
    for t_idx in range(NUM_TASKS):
        # Dynamically set Learning Rate: Task 0 vs Subsequent Tasks
        current_lr = lr_task0 if t_idx == 0 else LR_SUBSEQUENT
        for param_group in optimizer.param_groups:
            param_group["lr"] = current_lr

        current_perm = task_permutations[t_idx]

        tx = train_imgs[:, current_perm]
        ty = train_lbls

        print(
            f"\n--- Task {t_idx:02d}/{NUM_TASKS:02d} | Seed: {FIXED_SEED} | Active LR: {current_lr:.1e} ---"
        )

        for epoch in range(EPOCHS_PER_TASK):
            model.train()
            indices = torch.randperm(len(tx))

            for i in range(0, len(tx), BATCH_SIZE):
                batch_idx = indices[i : i + BATCH_SIZE]
                bx, by = tx[batch_idx], ty[batch_idx]

                optimizer.zero_grad()

                output = model(bx)
                loss_current = criterion(output, by)
                loss_current.backward()

                gpm.project_model_gradients(model)
                optimizer.step()

                # Snapshot logging evaluation cycle
                if total_steps % EVAL_EVERY_N_BATCHES == 0:
                    current_accs = []
                    model.eval()
                    with torch.no_grad():
                        for eval_t_idx in range(t_idx + 1):
                            eval_perm = task_permutations[eval_t_idx]
                            test_x = test_imgs_raw[:, eval_perm]

                            outputs = model(test_x[:1000])
                            preds = outputs.argmax(dim=1)
                            acc = (
                                (preds == test_lbls[:1000])
                                .float()
                                .mean()
                                .item()
                            )
                            current_accs.append(acc)

                    # Log task accuracy along with per-layer basis ranks and total rank
                    tracker.log(
                        step=total_steps,
                        acc_list=current_accs,
                        basis_rank=gpm.get_basis_ranks(),
                        total_basis_rank=gpm.get_total_basis_rank(),
                    )

                    acc_report = " | ".join(
                        [
                            f"T{j}: {current_accs[j] * 100:.1f}%"
                            for j in range(t_idx + 1)
                        ]
                    )
                    print(
                        f"LR_T0 {lr_task0:.5g} | Step {total_steps:04d} (Epoch {epoch}) -> {acc_report} | Total Basis Rank: {gpm.get_total_basis_rank()}"
                    )

                total_steps += 1

        # --- 5. POST-TASK CALIBRATION STEP ---
        model.eval()
        calib_indices = torch.randperm(len(tx))[:CALIB_SAMPLE_SIZE]
        calib_images = tx[calib_indices]

        with torch.no_grad():
            layer_inputs_dict = model.get_layer_inputs(calib_images)

        for (weight_param_name, _), (layer_key, layer_input) in zip(
            linear_layer_info, layer_inputs_dict.items()
        ):
            added_rank = gpm.update_basis(
                layer_id=weight_param_name,
                live_activations=layer_input,
                masking_fn=CUSTOM_MASKING_FN,
            )

        print(
            f"Task {t_idx} complete. Subspace memory updated. Current total basis rank: {gpm.get_total_basis_rank()}"
        )

    # --- 6. SERIALIZATION ---
    output_filepath = OUTPUT_DIR / f"gpm_a{ALPHA_INIT}_lr_{lr_task0:.5g}.csv"
    tracker.save_to_csv(output_filepath)
    print(
        f"Sweep for LR_Task0 = {lr_task0:.5g} serialized successfully to {output_filepath}.\n"
    )

print("\n" + "=" * 80)
print("ALL TASK 0 LEARNING RATE SWEEPS COMPLETED SUCCESSFULLY.")
print("=" * 80)

In [ ]:
import csv
import re
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Directory containing the sweep CSV files
SWEEP_DIR = Path("./task1_lr_fine_sweep")

lrs = []
avg_accs = []
hidden_ranks = []
total_ranks = []

# Regex to extract the floating-point LR from filenames (e.g., gpm_a1.2_lr_0.01585.csv)
lr_pattern = re.compile(r"_lr_([\d\.eE+-]+)\.csv$")

csv_files = sorted(list(SWEEP_DIR.glob("*.csv")))

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in {SWEEP_DIR.resolve()}. Make sure the sweep has completed."
    )

for file_path in csv_files:
    match = lr_pattern.search(file_path.name)
    if not match:
        continue

    lr_val = float(match.group(1))

    # Read the final line of the CSV
    with open(file_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]
        if not lines:
            continue
        last_line_str = lines[-1].split(",")

    values = [float(x) for x in last_line_str]

    # --- CSV ROW INDEX MAP ---
    # values[0]       -> Step (23400)
    # values[1:21]    -> Task 0 to Task 19 Test Accuracies (20 values)
    # values[21]      -> Total Basis Rank
    # values[22]      -> Layer 1 Basis Rank
    # values[23:32]   -> Layers 2 through 10 Basis Ranks (9 hidden layers)

    accs = values[1:21]
    avg_acc = np.mean(accs)

    total_rank = values[21]
    layer1_rank = values[22]
    hidden_rank = total_rank - layer1_rank  # Sum of Layers 2 to 10

    lrs.append(lr_val)
    avg_accs.append(avg_acc)
    hidden_ranks.append(hidden_rank)
    total_ranks.append(total_rank)

# Sort all data points in ascending order of LR for smooth curve rendering
sorted_indices = np.argsort(lrs)
lrs = np.array(lrs)[sorted_indices]
avg_accs = np.array(avg_accs)[sorted_indices]
hidden_ranks = np.array(hidden_ranks)[sorted_indices]
total_ranks = np.array(total_ranks)[sorted_indices]

# --- DUAL-AXIS RESPONSE SURFACE PLOT ---
fig, ax1 = plt.subplots(figsize=(10, 6), dpi=150)

# X-Axis: Log Scale Learning Rate
ax1.set_xscale("log")
ax1.set_xlabel(
    "Task 0 Learning Rate (Log Scale)", fontsize=12, fontweight="bold"
)

# Left Y-Axis: Global 20-Task Average Accuracy
color_acc = "#d62728"  # Red
ax1.set_ylabel(
    "20-Task Mean Accuracy", color=color_acc, fontsize=12, fontweight="bold"
)
line1 = ax1.plot(
    lrs,
    avg_accs,
    color=color_acc,
    marker="o",
    linewidth=2.5,
    label="20-Task Avg Accuracy",
)
ax1.tick_params(axis="y", labelcolor=color_acc)
ax1.grid(True, which="both", linestyle="--", alpha=0.3)

# Right Y-Axis: Hidden Layers Basis Rank (Layers 2-10)
ax2 = ax1.twinx()
color_rank = "#1f77b4"  # Blue
ax2.set_ylabel(
    "Hidden Basis Rank (Layers 2–10)",
    color=color_rank,
    fontsize=12,
    fontweight="bold",
)
line2 = ax2.plot(
    lrs,
    hidden_ranks,
    color=color_rank,
    marker="s",
    linewidth=2.5,
    linestyle="--",
    label="Hidden Basis Rank (L2–L10)",
)
ax2.tick_params(axis="y", labelcolor=color_rank)

# Combined Legend
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(
    lines, labels, loc="lower left", frameon=True, facecolor="white", framealpha=0.9
)

plt.title(
    "Task 0 Learning Rate Response Surface:\nAccuracy vs. Hidden Layer Basis Rank",
    fontsize=14,
    fontweight="bold",
    pad=15,
)

plt.tight_layout()

# Save high-resolution plot figure
output_plot_path = SWEEP_DIR / "task0_lr_response_surface.png"
plt.savefig(output_plot_path, dpi=300, bbox_inches="tight")
plt.show()

# --- PRINT CONSOLE SUMMARY TABLE ---
print("\n" + "=" * 65)
print(
    f"{'LR Task 0':<12} | {'Avg Acc (%)':<12} | {'Hidden Rank':<12} | {'Total Rank':<12}"
)
print("=" * 65)
for lr, acc, h_rank, t_rank in zip(lrs, avg_accs, hidden_ranks, total_ranks):
    print(
        f"{lr:<12.5g} | {acc*100:<12.2f} | {h_rank:<12.0f} | {t_rank:<12.0f}"
    )
print("=" * 65)
print(f"Plot saved successfully to: {output_plot_path.resolve()}")

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- 1. CONFIGURATION ---
GAUSSIAN_CSV = "gpm_a2.0_run_s0.csv"  # Standard Gaussian initialization run
HEAVY_TAIL_CSV = "gpm_a1.2_run_s0.csv"  # Heavy-Tailed initialization run
NUM_TASKS = 20


# --- 2. EXTRACTOR FUNCTION FOR A SINGLE CSV FILE ---
def extract_run_metrics(filepath, num_tasks=20):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Could not find target CSV file at: '{filepath}'")

    df = pd.read_csv(filepath)
    num_entries = len(df)

    # Sort chronological accuracy columns
    task_acc_cols = [
        c for c in df.columns if c.startswith("task_") and c.endswith("_acc")
    ]
    task_acc_cols = sorted(task_acc_cols, key=lambda x: int(x.split("_")[1]))

    # Determine active task footprint per row to find task completion boundaries
    active_tasks_per_step = np.array(
        [df.iloc[idx][task_acc_cols].notna().sum() for idx in range(num_entries)]
    )

    # Detect transition step indices where each task finishes training
    transition_indices = []
    current_num = active_tasks_per_step[0]
    for idx in range(1, num_entries):
        if active_tasks_per_step[idx] > current_num:
            transition_indices.append(idx - 1)
            current_num = active_tasks_per_step[idx]
    transition_indices.append(num_entries - 1)

    if len(transition_indices) != num_tasks:
        print(
            f"Warning: Boundary mismatch in {filepath}. Detected {len(transition_indices)} boundaries instead of {num_tasks}"
        )

    # Determine the step intervals for each task
    task_step_intervals = []
    start_idx = 0
    for end_idx in transition_indices:
        task_step_intervals.append((start_idx, end_idx))
        start_idx = end_idx + 1

    # Storage arrays
    avg_acc_at_wrapup = []
    current_task_auc = []
    total_basis_rank = []
    layer_1_basis_rank = []

    # Detect available rank columns dynamically
    layer_1_col = "basis_rank_0" if "basis_rank_0" in df.columns else None
    if layer_1_col is None:
        layer_1_cols = [
            c
            for c in df.columns
            if "basis" in c and ("0" in c or "layer_0" in c or "fc1" in c)
        ]
        layer_1_col = layer_1_cols[0] if len(layer_1_cols) > 0 else None

    for t_idx, boundary_idx in enumerate(transition_indices):
        # 1. Global Average Accuracy across all active tasks
        row_accs = df.iloc[boundary_idx][task_acc_cols].values
        active_accs = row_accs[~pd.isna(row_accs)]
        avg_acc_at_wrapup.append(np.mean(active_accs) if len(active_accs) > 0 else 0.0)

        # 2. Plasticity AUC during training of Task t_idx
        start_step_idx, end_step_idx = task_step_intervals[t_idx]
        task_col = f"task_{t_idx}_acc"

        task_trajectory = (
            df.iloc[start_step_idx : end_step_idx + 1][task_col].dropna().values
        )
        steps = (
            df.iloc[start_step_idx : end_step_idx + 1]["step"]
            .iloc[: len(task_trajectory)]
            .values
        )

        if len(task_trajectory) > 1:
            auc_val = np.trapezoid(y=task_trajectory, x=steps) / (steps[-1] - steps[0])
        elif len(task_trajectory) == 1:
            auc_val = task_trajectory[0]
        else:
            auc_val = 0.0
        current_task_auc.append(auc_val)

        # 3. Total Basis Rank at task wrap-up
        if "total_basis_rank" in df.columns:
            total_rank_val = df.iloc[boundary_idx]["total_basis_rank"]
        else:
            rank_cols = [c for c in df.columns if c.startswith("basis_rank_")]
            total_rank_val = (
                df.iloc[boundary_idx][rank_cols].sum() if len(rank_cols) > 0 else np.nan
            )
        total_basis_rank.append(total_rank_val)

        # 4. Layer 1 Basis Rank at task wrap-up
        if layer_1_col and layer_1_col in df.columns:
            layer_1_val = df.iloc[boundary_idx][layer_1_col]
        else:
            layer_1_val = np.nan
        layer_1_basis_rank.append(layer_1_val)

    return {
        "avg_acc": avg_acc_at_wrapup,
        "auc": current_task_auc,
        "total_rank": total_basis_rank,
        "layer1_rank": layer_1_basis_rank,
    }


# --- 3. PROCESS BOTH RUNS ---
print(f"Extracting Gaussian baseline metrics from: {GAUSSIAN_CSV}")
gauss_data = extract_run_metrics(GAUSSIAN_CSV, NUM_TASKS)

print(f"Extracting Heavy-Tailed metrics from: {HEAVY_TAIL_CSV}")
ht_data = extract_run_metrics(HEAVY_TAIL_CSV, NUM_TASKS)


# --- 4. 2x2 COMPARATIVE VISUALIZATION ---
fig, axs = plt.subplots(2, 2, figsize=(15, 11), dpi=100)
tasks_axis = np.arange(1, NUM_TASKS + 1)

# Color and style definitions
gauss_color, ht_color = "crimson", "dodgerblue"
gauss_marker, ht_marker = "o", "s"

# [TOP LEFT] Global Cumulative Average Accuracy
axs[0, 0].plot(
    tasks_axis,
    gauss_data["avg_acc"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[0, 0].plot(
    tasks_axis,
    ht_data["avg_acc"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Gaussian",
)
axs[0, 0].set_title("1. Global Average Test Accuracy", fontsize=11, weight="bold")
axs[0, 0].set_xlabel("Task Index")
axs[0, 0].set_ylabel("Mean Accuracy across Learned Tasks")
axs[0, 0].set_xticks(tasks_axis)
axs[0, 0].grid(True, linestyle=":", alpha=0.5)
axs[0, 0].legend()

# [TOP RIGHT] Task Plasticity AUC
axs[0, 1].plot(
    tasks_axis,
    gauss_data["auc"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[0, 1].plot(
    tasks_axis,
    ht_data["auc"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Gaussian",
)
axs[0, 1].set_title(
    "2. Current Task Learning Curve AUC (Plasticity)", fontsize=11, weight="bold"
)
axs[0, 1].set_xlabel("Task Index")
axs[0, 1].set_ylabel("Normalized Training AUC")
axs[0, 1].set_xticks(tasks_axis)
axs[0, 1].grid(True, linestyle=":", alpha=0.5)
axs[0, 1].legend()

# [BOTTOM LEFT] Total Basis Rank
axs[1, 0].plot(
    tasks_axis,
    gauss_data["total_rank"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[1, 0].plot(
    tasks_axis,
    ht_data["total_rank"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Gaussian",
)
axs[1, 0].set_title("3. Cumulative Total Basis Rank", fontsize=11, weight="bold")
axs[1, 0].set_xlabel("Task Index")
axs[1, 0].set_ylabel("Total Reserved Basis Vectors")
axs[1, 0].set_xticks(tasks_axis)
axs[1, 0].grid(True, linestyle=":", alpha=0.5)
axs[1, 0].legend()

# [BOTTOM RIGHT] Layer 1 Basis Rank
axs[1, 1].plot(
    tasks_axis,
    gauss_data["layer1_rank"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[1, 1].plot(
    tasks_axis,
    ht_data["layer1_rank"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Gaussian",
)
axs[1, 1].set_title("4. Layer 1 Basis Rank", fontsize=11, weight="bold")
axs[1, 1].set_xlabel("Task Index")
axs[1, 1].set_ylabel("Layer 1 Reserved Basis Vectors")
axs[1, 1].set_xticks(tasks_axis)
axs[1, 1].grid(True, linestyle=":", alpha=0.5)
axs[1, 1].legend()

plt.tight_layout()
plt.savefig("gpm_gaussian_vs_heavytail_comparison.pdf", bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path

import torch
from torch import nn
from torchvision import datasets, transforms

# --- 1. CONFIGURATION matching your training script ---
SNAPSHOT_PATH = Path("./checkpoints/snapshot_A2.0_T20_E4.pt")
NUM_TASKS = 20
BATCH_SIZE = 1024  # Size of the evaluation batch per task

# Architecture hyper-parameters
HIDDEN_SIZE = 784
DEPTH = 9
ACTIVATION_NAME = "tanh"
BIAS = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. LOAD SNAPSHOT & METADATA ---
if not SNAPSHOT_PATH.exists():
    raise FileNotFoundError(f"Snapshot not found at: {SNAPSHOT_PATH}")

print(f"Loading snapshot from: {SNAPSHOT_PATH}")
snapshot = torch.load(SNAPSHOT_PATH, map_location=DEVICE)

metadata = snapshot.get("metadata", {})
seed = metadata.get("seed", 0)  # Defaults to 0 if seed wasn't explicitly saved
print(
    f"Snapshot Metadata -> Task: {metadata.get('task')}, Epoch: {metadata.get('epoch')}, Seed: {seed}"
)

# --- 3. RECONSTRUCT MODEL & LOAD WEIGHTS ---
model = GeneralMLP(
    input_size=784,
    hidden_size=HIDDEN_SIZE,
    num_classes=10,
    depth=DEPTH,
    activation_name=ACTIVATION_NAME,
    bias=BIAS,
).to(DEVICE)

model.load_state_dict(snapshot["state_dict"])
model.eval()
print("Model architecture successfully reconstructed and state_dict loaded.")

# --- 4. REGENERATE PERMUTATIONS & TEST DATA ---
set_seed(seed)
task_permutations = generate_permutations(num_tasks=NUM_TASKS, seed=seed)

mnist_test = datasets.MNIST(
    "../data", train=False, download=True, transform=transforms.ToTensor()
)
test_imgs_raw, _ = get_gpu_data(mnist_test)  # Raw unpermuted GPU images [N, 784]

# --- 5. EXTRACT PRE-ACTIVATIONS PER LAYER PER TASK ---
# Result structure: task_pre_acts[t_idx][layer_idx] -> Tensor of shape [BATCH_SIZE, layer_dim]
task_pre_acts = []

with torch.no_grad():
    for t_idx in range(NUM_TASKS):
        perm = task_permutations[t_idx]
        task_batch = test_imgs_raw[
            :BATCH_SIZE, perm
        ]  # Permute input batch for task t_idx

        # Retrieve layer pre-activations using your GeneralMLP model helper
        # Returns a dict or list depending on implementation
        pre_acts = model.get_pre_activations(task_batch)

        # Convert dictionary format to a standardized ordered list per layer
        if isinstance(pre_acts, dict):
            # Extract linear hidden layers followed by classifier
            layer_keys = [k for k in pre_acts.keys() if k != "classifier"]
            layer_list = [pre_acts[k] for k in layer_keys]
            if "classifier" in pre_acts:
                layer_list.append(pre_acts["classifier"])
        else:
            layer_list = pre_acts

        task_pre_acts.append(layer_list)
        print(
            f"Captured pre-activations for Task {t_idx:02d} | Layers captured: {len(layer_list)}"
        )

print("\nPre-activation extraction complete.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib import cm


def convert_pre_to_post_activations(task_pre_acts, act_fn=torch.tanh):
    """Converts a nested list of pre-activations [num_tasks][num_layers]
    into 1D mean absolute post-activations [hidden_dim] per task and layer.
    """
    task_post_acts = []
    for t_idx in range(len(task_pre_acts)):
        layer_posts = []
        for layer_pre in task_pre_acts[t_idx]:
            post_act = act_fn(layer_pre)
            mean_abs_act = torch.abs(post_act).mean(dim=0)
            layer_posts.append(mean_abs_act.cpu().numpy())
        task_post_acts.append(layer_posts)
    return task_post_acts


def compute_all_pairs_energy_jaccard(task_post_acts, layer_idx, energy_fractions):
    """Computes Energy-Gated Soft Jaccard across all N*(N-1)/2 task pairs
    for a given layer index.
    """
    num_tasks = len(task_post_acts)
    task_acts = [task_post_acts[t][layer_idx] for t in range(num_tasks)]

    pair_indices = [(i, j) for i in range(num_tasks) for j in range(i + 1, num_tasks)]
    num_pairs = len(pair_indices)

    all_jaccard_curves = np.zeros((num_pairs, len(energy_fractions)))
    all_k_curves = np.zeros((num_pairs, len(energy_fractions)))

    print(f"Processing {num_pairs} pairwise comparisons for Layer {layer_idx + 1}...")

    for p_idx, (i, j) in enumerate(pair_indices):
        act_A, act_B = task_acts[i], task_acts[j]

        energy_A, energy_B = act_A**2, act_B**2
        total_E_A, total_E_B = np.sum(energy_A), np.sum(energy_B)

        sort_idx_A = np.argsort(energy_A)[::-1]
        sort_idx_B = np.argsort(energy_B)[::-1]

        cumsum_E_A = np.cumsum(energy_A[sort_idx_A]) / (total_E_A + 1e-12)
        cumsum_E_B = np.cumsum(energy_B[sort_idx_B]) / (total_E_B + 1e-12)

        for e_idx, target_E in enumerate(energy_fractions):
            k_A = min(np.searchsorted(cumsum_E_A, target_E) + 1, len(act_A))
            k_B = min(np.searchsorted(cumsum_E_B, target_E) + 1, len(act_B))

            gated_A = np.zeros_like(act_A)
            gated_B = np.zeros_like(act_B)

            gated_A[sort_idx_A[:k_A]] = act_A[sort_idx_A[:k_A]]
            gated_B[sort_idx_B[:k_B]] = act_B[sort_idx_B[:k_B]]

            num = np.sum(np.minimum(gated_A, gated_B))
            den = np.sum(np.maximum(gated_A, gated_B))

            all_jaccard_curves[p_idx, e_idx] = num / den if den > 0 else 0.0
            all_k_curves[p_idx, e_idx] = (k_A + k_B) / 2.0

    return all_jaccard_curves, all_k_curves


def run_bootstrap_ci(data_matrix, num_bootstraps=1000, ci_level=95):
    """Computes bootstrap mean and percentile confidence intervals along axis 0."""
    num_pairs, num_points = data_matrix.shape
    boot_means = np.zeros((num_bootstraps, num_points))

    rng = np.random.default_rng(seed=42)
    for b in range(num_bootstraps):
        boot_indices = rng.choice(num_pairs, size=num_pairs, replace=True)
        boot_means[b, :] = np.mean(data_matrix[boot_indices, :], axis=0)

    lower_p = (100 - ci_level) / 2.0
    upper_p = 100 - lower_p

    mean_curve = np.mean(data_matrix, axis=0)
    ci_lower = np.percentile(boot_means, lower_p, axis=0)
    ci_upper = np.percentile(boot_means, upper_p, axis=0)

    return mean_curve, ci_lower, ci_upper


# --- EXECUTION & MULTI-LAYER PLOTTING ROUTINE ---
ACTIVATION_FN = torch.tanh  # Or torch.relu for ReLU runs
task_post_acts = convert_pre_to_post_activations(task_pre_acts, act_fn=ACTIVATION_FN)

# 1. SELECT TARGET LAYERS TO COMPARE ACROSS DEPTH (0-indexed)
# For a 9-layer MLP, selecting input, early, mid, late, and classifier layers
TARGET_LAYERS = [0, 2, 4, 6, 8]
SHOW_CONFIDENCE_INTERVALS = True  # Toggle shaded CI bands on/off for readability

energy_fractions = np.linspace(0.02, 1.0, 99)
x_axis = energy_fractions * 100

# Generate color spectrum from blue (shallow) to red/purple (deep)
colors = cm.plasma(np.linspace(0.1, 0.9, len(TARGET_LAYERS)))

fig, ax1 = plt.subplots(figsize=(12, 7), dpi=100)
ax2 = ax1.twinx()

for idx, l_idx in enumerate(TARGET_LAYERS):
    # Process 190 task pairs for layer l_idx
    j_matrix, k_matrix = compute_all_pairs_energy_jaccard(
        task_post_acts, l_idx, energy_fractions
    )

    # Perform Bootstrapping
    j_mean, j_low, j_high = run_bootstrap_ci(j_matrix, num_bootstraps=1000)
    k_mean, k_low, k_high = run_bootstrap_ci(k_matrix, num_bootstraps=1000)

    color = colors[idx]
    layer_label = f"Layer {l_idx + 1}"

    # Primary Axis: Soft Jaccard Overlap
    ax1.plot(
        x_axis,
        j_mean,
        color=color,
        linewidth=2.5,
        label=f"{layer_label} Jaccard",
    )

    if SHOW_CONFIDENCE_INTERVALS:
        ax1.fill_between(x_axis, j_low, j_high, color=color, alpha=0.1)

    # Secondary Axis: Active Neurons (k)
    # ax2.plot(
    #     x_axis,
    #     k_mean,
    #     color=color,
    #     linestyle="--",
    #     linewidth=1.5,
    #     alpha=0.7,
    #     label=f"{layer_label} Active k",
    # )

# Axis Formatting
ax1.set_xlabel("Cumulative Task Activation Energy Mass (%)", fontsize=11, weight="bold")
ax1.set_ylabel("Continuous Soft Jaccard Overlap Score", fontsize=11, weight="bold")
ax1.set_xlim(0, 100)
ax1.set_ylim(0, 1.0)
ax1.grid(True, linestyle=":", alpha=0.6)

ax2.set_ylabel(
    "Mean Active Neurons Required (k)", color="gray", fontsize=11, weight="bold"
)

plt.title(
    "Depth-Wise Energy-Gated Soft Jaccard Evolution (190 Task Pairs)\nComparing Representation Compression & Overlap Across Layers",
    fontsize=12,
    weight="bold",
)

# Combine legends cleanly
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left", fontsize=9, ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import skdim
import torch
from scipy.special import digamma
from sklearn.neighbors import NearestNeighbors


# --- 1. MATHEMATICALLY ALIGNED ESTIMATORS ---
def compute_global_svd_rank(X, variance_threshold=0.95):
    """Computes global linear rank using SVD at a cumulative variance threshold."""
    # Center activations
    X_centered = X - np.mean(X, axis=0, keepdims=True)

    # Singular Value Decomposition
    _, S, _ = np.linalg.svd(X_centered, full_matrices=False)

    # Calculate cumulative variance fraction
    variance_explained = (S**2) / np.sum(S**2)
    cumsum_var = np.cumsum(variance_explained)

    # Find number of components to reach variance_threshold
    rank = np.searchsorted(cumsum_var, variance_threshold) + 1
    return min(rank, X.shape[1])


def compute_local_pca_id(X, k=30, variance_threshold=0.95):
    """Computes Local PCA ID across k-NN neighborhoods matching the 95% variance threshold."""
    # ver='ratio' forces lPCA to use cumulative variance thresholding (alphaRatio)
    lpca_model = skdim.id.lPCA(ver="ratio", alphaRatio=variance_threshold)

    # Fit pointwise across k-NN neighborhoods
    lpca_model.fit_pw(X, n_neighbors=k)

    # Average local tangent rank across all neighborhood balls
    return np.mean(lpca_model.dimension_pw_)


def compute_gride_id(X, k1=10, k2=20):
    """Computes non-linear Intrinsic Dimension using GRIDE (Generalized Ratio Estimator).

    MLE formula: d = [digamma(k2) - digamma(k1)] / mean(log(r_k2 / r_k1))
    """
    # Compute k2 nearest neighbors for each point
    nbrs = NearestNeighbors(n_neighbors=k2 + 1, algorithm="auto").fit(X)
    distances, _ = nbrs.kneighbors(X)

    # Extract distances to k1-th and k2-th neighbors
    r1 = distances[:, k1]
    r2 = distances[:, k2]

    # Filter out identical or zero distances
    valid_mask = (r1 > 1e-12) & (r2 > r1)
    if not np.any(valid_mask):
        return 0.0

    log_ratios = np.log(r2[valid_mask] / r1[valid_mask])
    mean_log_ratio = np.mean(log_ratios)

    # Digamma difference numerator
    digamma_diff = digamma(k2) - digamma(k1)

    return digamma_diff / mean_log_ratio


def compute_twonn_id(X):
    """Optional alternative: Native skdim TwoNN estimator (k1=1, k2=2)."""
    twonn = skdim.id.TwoNN()
    twonn.fit(X)
    return twonn.dimension_


# --- 2. EXECUTION ROUTINE ACROSS ALL LAYERS ---
TARGET_TASK_IDX = 1  # Task 0 for clean baseline geometry
VARIANCE_THRESHOLD = 0.95  # 95% energy mass threshold
ACTIVATION_FN = torch.tanh  # Post-activations

num_layers = len(task_pre_acts[TARGET_TASK_IDX])

global_svd_ranks = []
local_pca_ids = []
gride_ids = []

print(
    f"Evaluating Dimension Metrics for Task {TARGET_TASK_IDX} across {num_layers} layers..."
)

for l_idx in range(num_layers):
    layer_pre = task_pre_acts[TARGET_TASK_IDX][l_idx]

    # Convert to post-activations: [N_samples, hidden_dim]
    if isinstance(layer_pre, torch.Tensor):
        post_act_samples = ACTIVATION_FN(layer_pre).detach().cpu().numpy()
    else:
        post_act_samples = ACTIVATION_FN(torch.tensor(layer_pre)).numpy()

    print(
        f"--> Processing Layer {l_idx + 1}/{num_layers} (Matrix Shape: {post_act_samples.shape})..."
    )

    # 1. Global SVD Rank (95% Variance)
    g_svd = compute_global_svd_rank(
        post_act_samples, variance_threshold=VARIANCE_THRESHOLD
    )

    # 2. Local PCA ID (Local 95% Tangent Rank in 30-NN neighborhood)
    l_pca = compute_local_pca_id(
        post_act_samples, k=30, variance_threshold=VARIANCE_THRESHOLD
    )

    # 3. GRIDE ID (Noise-filtered non-linear manifold dimension)
    gride = compute_gride_id(post_act_samples, k1=10, k2=20)

    global_svd_ranks.append(g_svd)
    local_pca_ids.append(l_pca)
    gride_ids.append(gride)

    print(
        f"    Global SVD (95%): {g_svd} | Local PCA (95%): {l_pca:.1f} | GRIDE: {gride:.1f}"
    )


# --- 3. PLOTTING LAYERWISE COMPARISON CURVE ---
layers_x = np.arange(1, num_layers + 1)

plt.figure(figsize=(10, 6), dpi=100)

plt.plot(
    layers_x,
    global_svd_ranks,
    marker="o",
    color="darkred",
    linewidth=2.5,
    markersize=7,
    label=f"Global SVD Rank ({int(VARIANCE_THRESHOLD * 100)}% Variance)",
)

plt.plot(
    layers_x,
    local_pca_ids,
    marker="s",
    color="darkorange",
    linewidth=2.5,
    markersize=7,
    linestyle="--",
    label=f"Local PCA ID (Local {int(VARIANCE_THRESHOLD * 100)}% Tangent Rank)",
)

plt.plot(
    layers_x,
    gride_ids,
    marker="^",
    color="dodgerblue",
    linewidth=2.5,
    markersize=7,
    linestyle="-.",
    label="GRIDE ID (Non-Linear Manifold Dimension, k1=10, k2=20)",
)

plt.title(
    f"Layerwise Intrinsic Dimensionality & Linear Rank Profile\nTask {TARGET_TASK_IDX} Post-Activations | Depth-Wise Manifold Trajectory",
    fontsize=12,
    weight="bold",
)
plt.xlabel("Layer Index (Depth)", fontsize=11, weight="bold")
plt.ylabel("Estimated Dimension / Rank", fontsize=11, weight="bold")
plt.xticks(layers_x)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper right", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch


# --- 1. SUBSPACE EXTRACTION AND METRIC FUNCTIONS ---
def extract_task_basis(X, variance_threshold=0.95):
    """Extracts the top-k right singular vectors V_k representing the feature subspace

    that accounts for the given cumulative variance threshold.
    X shape: [N_samples, hidden_dim]
    """
    # Center activations across samples
    X_centered = X - np.mean(X, axis=0, keepdims=True)

    # Compute SVD: X = U @ S @ Vt
    _, S, Vt = np.linalg.svd(X_centered, full_matrices=False)

    # Compute required rank k
    var_explained = (S**2) / np.sum(S**2)
    cumsum_var = np.cumsum(var_explained)
    k = np.searchsorted(cumsum_var, variance_threshold) + 1
    k = min(k, X.shape[1])

    # Top-k feature basis vectors in ambient space: shape [hidden_dim, k]
    V_k = Vt[:k, :].T
    return V_k, X_centered


def compute_normalized_subspace_overlap(V_A, V_B):
    """Method A: Normalized Subspace Overlap Score ||V_A^T V_B||_F^2 / min(k_A, k_B)

    Bounded in [0.0, 1.0].
    """
    k_A, k_B = V_A.shape[1], V_B.shape[1]
    projection_matrix = V_A.T @ V_B  # [k_A, k_B]
    frobenius_sq = np.linalg.norm(projection_matrix, ord="fro") ** 2
    return frobenius_sq / min(k_A, k_B)


def compute_unexplained_variance_fraction(X_B_centered, V_A):
    """Method C: Fraction of Task B's energy that lies OUTSIDE Task A's subspace.

    Mirrors GPM's memory allocation check. Bounded in [0.0, 1.0].
    """
    # Project Task B onto Task A's basis subspace
    projected_B = X_B_centered @ V_A @ V_A.T
    explained_energy = np.linalg.norm(projected_B, ord="fro") ** 2
    total_energy = np.linalg.norm(X_B_centered, ord="fro") ** 2 + 1e-12

    unexplained_fraction = 1.0 - (explained_energy / total_energy)
    return max(0.0, unexplained_fraction)


# --- 2. EXECUTION ROUTINE ACROSS ALL LAYERS ---
VARIANCE_THRESHOLD = 0.95  # 95% energy threshold
ACTIVATION_FN = torch.tanh  # Post-activations

num_tasks = len(task_pre_acts)
num_layers = len(task_pre_acts[0])
task_pairs = list(itertools.combinations(range(num_tasks), 2))
num_pairs = len(task_pairs)

mean_overlaps_per_layer = []
mean_unexplained_per_layer = []

print(
    f"Evaluating Subspace Overlap across {num_layers} layers for {num_pairs} task pairs..."
)

for l_idx in range(num_layers):
    # 1. Extract post-activations and bases for all tasks at layer l_idx
    task_bases = []
    task_acts_centered = []

    for t_idx in range(num_tasks):
        layer_pre = task_pre_acts[t_idx][l_idx]
        if isinstance(layer_pre, torch.Tensor):
            post_act = ACTIVATION_FN(layer_pre).detach().cpu().numpy()
        else:
            post_act = ACTIVATION_FN(torch.tensor(layer_pre)).numpy()

        V_k, X_centered = extract_task_basis(
            post_act, variance_threshold=VARIANCE_THRESHOLD
        )
        task_bases.append(V_k)
        task_acts_centered.append(X_centered)

    # 2. Compute metrics across all 190 unique pairs (A, B)
    pair_overlaps = []
    pair_unexplained = []

    for t_A, t_B in task_pairs:
        V_A, V_B = task_bases[t_A], task_bases[t_B]
        X_B_centered = task_acts_centered[t_B]

        overlap = compute_normalized_subspace_overlap(V_A, V_B)
        unexplained = compute_unexplained_variance_fraction(X_B_centered, V_A)

        pair_overlaps.append(overlap)
        pair_unexplained.append(unexplained)

    layer_mean_overlap = np.mean(pair_overlaps)
    layer_mean_unexplained = np.mean(pair_unexplained)

    mean_overlaps_per_layer.append(layer_mean_overlap)
    mean_unexplained_per_layer.append(layer_mean_unexplained)

    print(
        f"--> Layer {l_idx + 1:2d}/{num_layers:2d} | Subspace Overlap: {layer_mean_overlap:.4f} | GPM Unexplained Energy: {layer_mean_unexplained:.4f}"
    )


# --- 3. PLOTTING THE TRAJECTORY ACROSS DEPTH ---
layers_x = np.arange(1, num_layers + 1)

fig, ax1 = plt.subplots(figsize=(10, 6), dpi=100)

color_overlap = "dodgerblue"
ax1.plot(
    layers_x,
    mean_overlaps_per_layer,
    marker="o",
    color=color_overlap,
    linewidth=2.5,
    markersize=7,
    label="Normalized Subspace Overlap Score",
)
ax1.set_xlabel("Layer Index (Depth)", fontsize=11, weight="bold")
ax1.set_ylabel(
    "Mean Subspace Overlap (190 Pairs)",
    color=color_overlap,
    fontsize=11,
    weight="bold",
)
ax1.set_ylim(0, 1.0)
ax1.grid(True, linestyle=":", alpha=0.6)

ax2 = ax1.twinx()
color_unexplained = "crimson"
ax2.plot(
    layers_x,
    mean_unexplained_per_layer,
    marker="s",
    color=color_unexplained,
    linewidth=2.5,
    markersize=7,
    linestyle="--",
    label="GPM Unexplained Energy Fraction",
)
ax2.set_ylabel(
    "Mean Unexplained Energy Fraction",
    color=color_unexplained,
    fontsize=11,
    weight="bold",
)
ax2.set_ylim(0, 1.0)

plt.title(
    f"Cross-Task Subspace Alignment Profile Across Depth ({num_tasks} Tasks, 190 Pairs)\nEvaluating SVD Subspace Coherence & GPM Expansion Need",
    fontsize=12,
    weight="bold",
)
plt.xticks(layers_x)

# Combine legends
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="center left")

plt.tight_layout()
plt.show()

In [ ]:
# --- Configuration ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TASKS = [[0, 1], [2, 3], [4, 5], [6, 7], [8, 9]]
LR = 1e-2
GAMMA = 0.001
K_STEEPNESS = 3000.0
BATCH_SIZE = 64  # Increased for GPU efficiency
EPOCHS_PER_TASK = 10


class EbbinghausOptimizer:
    def __init__(self, model, lr=LR, gamma=GAMMA, k_max=K_STEEPNESS, k_min=10):
        self.params = [p for p in model.parameters() if p.requires_grad]
        self.lr = lr
        self.gamma = gamma
        self.k_max = k_max
        self.k_min = k_min
        self.k_layers = []
        for i in range(len(self.params)):
            # Linear example:
            ratio = i / (len(self.params) - 1)
            k_val = self.k_max - ratio * (self.k_max - self.k_min)
            self.k_layers.append(k_val)
        # Stability accumulator S_i
        self.stability = [torch.zeros_like(p, device=DEVICE) for p in self.params]

    def step(self):
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is None:
                    continue

                # 1. Dynamic Decay (Standard)
                # We can stick to a simpler decay here since the growth is self-regulating
                self.stability[i] = (1.0 - self.gamma) * self.stability[i]

                # 2. Saturating Growth (Diminishing Returns)
                # The 'fuller' the memory, the harder it is to add more.
                grad_sq = p.grad**2
                growth_resistance = 1.0 + self.stability[i]
                self.stability[i] += grad_sq / growth_resistance

                # 3. Calculate Hyperbolic Plasticity Filter
                plasticity = 1.0 / (1.0 + self.k_layers[i] * self.stability[i])

                # 4. Apply Filtered Update
                p.data -= self.lr * plasticity * p.grad

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.detach_()
                p.grad.zero_()


# --- 2. Training Loop ---
model = SimpleMLP().to(DEVICE)
ebbinghaus_opt = EbbinghausOptimizer(model)
criterion = nn.CrossEntropyLoss()
tracker = MetricsTracker()
total_steps = 0

(train_x, train_y), (test_x, test_y) = get_gpu_mnist()
all_test_tasks = [get_task_data(test_x, test_y, t) for t in TASKS]
results_matrix = []

for t_idx, digits in enumerate(TASKS):
    tx, ty = get_task_data(train_x, train_y, digits)
    print(f"Training Task {t_idx} (Digits {digits})")

    for epoch in range(EPOCHS_PER_TASK):
        indices = torch.randperm(len(tx))
        for i in range(0, len(tx), BATCH_SIZE):
            batch_idx = indices[i : i + BATCH_SIZE]
            bx, by = tx[batch_idx], ty[batch_idx]

            ebbinghaus_opt.zero_grad()
            output = model(bx)
            loss = criterion(output, by)
            loss.backward()

            # This is where the magic happens: No penalty, just a filtered step
            ebbinghaus_opt.step()

            total_steps += 1

            # High-Resolution Intra-Task Tracking
            if total_steps % EVAL_EVERY_N_BATCHES == 0:
                current_accs = evaluate(model, all_test_tasks)
                tracker.log(
                    total_steps, current_accs, loss.item(), ebbinghaus_opt.stability
                )

        current_accs = evaluate(model, all_test_tasks)
        print(f"Epoch {epoch} Accuracies: {[round(a, 2) for a in current_accs]}")

    results_matrix.append(current_accs)